# 7006SCN Task 1 — Problem, Data and Pipeline

**Student:** Manish Saini · **SID:** 17142113 · **Pool reference:** 112
**Dataset:** `climate.csv` — NOAA GHCN-Daily, US weather stations, 1970–2023, mirrored on Hugging Face as [`CatLikeIceCream/temperature`](https://huggingface.co/datasets/CatLikeIceCream/temperature)
**Licence:** none declared on the mirror; the original NOAA GHCN-Daily data is US-Government public-domain data (cite Menne et al., 2012)

## 1.1 Problem and why this dataset fits it

**Problem (REGRESSION):** predict tomorrow's maximum temperature (°C) at a US weather station from that station's recent observations. Next-day maximum temperature is used for decisions such as electricity-demand planning and heat or frost warnings.

**Target:** `label_TMAX_next` = the `TMAX` of the next calendar day at the same station, in °C (continuous).

**Why this pairing:** every station is a long daily series of `TMAX`, `TMIN`, `TOBS`, `PRCP`, `SNOW` and `SNWD`, so lagged, rolling and seasonal predictors can be built per station without using any future value. Table T1.2 (section 9) measures how strongly these predictors relate to the target.

**Limitation carried into later tasks:** the file has no latitude, longitude or elevation, so location enters only as a state code taken from the station ID; and stations start, stop and have gaps, so every lag feature must check that the earlier day really exists (section 9).



# 0.1 Environment (Google Colab or local Jupyter)
# REF: Week1_Day02_PySpark_Student_Workbook › Sub-task 0.1 (cell-1)



In [ ]:
# Windows-only setup for local Jupyter (skipped on Colab):
# - Hadoop helper files (winutils.exe, hadoop.dll) are needed for Spark to write Parquet on Windows
# - Spark temp files and lakehouse/model output go to E:, because C: was almost full and Spark crashed
import os
if os.name == "nt":
    os.environ["HADOOP_HOME"]  = r"E:\hadoop"
    os.environ["PATH"]         = r"E:\hadoop\bin;" + os.environ["PATH"]
    os.environ["CW_LAKE_DIR"]  = "E:/7006SCN/lakehouse"
    os.environ["CW_MODEL_DIR"] = "E:/7006SCN/models"
    os.environ["CW_LOCAL_DIR"] = "E:/spark-tmp"
    os.environ["TEMP"] = os.environ["TMP"] = r"E:\spark-tmp"
    os.makedirs(r"E:\spark-tmp", exist_ok=True)

In [22]:
import sys, os, time, json, math

IN_COLAB = 'google.colab' in sys.modules              # EXPLAIN: True only when running on Google Colab
if IN_COLAB:
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    ## AI: added huggingface_hub (dataset download) and matplotlib (Fig. T1.1)
    !pip install -q pyspark huggingface_hub matplotlib
else:
    try:
        import pyspark; print("Local PySpark", pyspark.__version__)
    except ImportError:
        !pip install pyspark huggingface_hub matplotlib

Local PySpark 4.2.0


0.2 Download climate.csv from Hugging Face

In [ ]:
# REF: Day02_LiveCoding_Colab › Block 0 (cell-5) downloaded with the Kaggle CLI.
# Local run from /notebooks: use the repo-root data folder, so the 2.73 GB file is not downloaded twice
LOCAL_DATA = "../data" if os.path.basename(os.getcwd()).lower() == "notebooks" else "./data"
DATA_DIR = os.environ.get("CW_DATA_DIR", "/content/data" if IN_COLAB else LOCAL_DATA)
CSV_PATH = os.path.join(DATA_DIR, "climate.csv")
os.makedirs(DATA_DIR, exist_ok=True)                   # EXPLAIN: create the folder if missing

if not os.path.exists(CSV_PATH):                       # EXPLAIN: skip the 2.73 GB download if already present
    from huggingface_hub import hf_hub_download        # EXPLAIN: downloads one file from a Hugging Face repo
    hf_hub_download(repo_id="CatLikeIceCream/temperature", filename="climate.csv",
                    repo_type="dataset", local_dir=DATA_DIR)
print("Data file:", CSV_PATH, "| exists:", os.path.exists(CSV_PATH))

1. Student Name and Dataset declaration

In [ ]:
DECLARATION = {
    "student_name"   : "Manish Saini",
    "student_email"  : "sainim10@coventry.ac.uk",
    "sid"            : "17142113",
    "pool_reference" : "112",
    "dataset_name"   : "climate.csv - GHCN-Daily US weather-station records (1970-2023)",
    "source_url"     : "https://huggingface.co/datasets/CatLikeIceCream/temperature",
    "original_source": "NOAA NCEI Global Historical Climatology Network - Daily (GHCN-Daily)",
    "licence"        : "None declared on the Hugging Face mirror; original NOAA GHCN-Daily is US-Government "
                       "public-domain data (citation of Menne et al. 2012 requested)",
    "ethics"         : "No personal data: records describe weather stations, not people. Risk = coverage bias "
                       "(uneven station density across states and decades).",
    "problem"        : "REGRESSION - predict next-day maximum temperature (deg C) per station",
}
for k, v in DECLARATION.items():
    print(f"{k:16s}: {v}")

## 2. SparkSession initialisation and cluster configuration
REF: Week1_Day02_PySpark_Student_Workbook › Exercise 0, Sub-tasks 0.1–0.2 (cell-5)
Changed: shuffle partitions 8 → 32. Driver memory is **4 g**, and Spark's temp files go to E:, because with 6 g the Java process crashed with "insufficient memory" on my 16 GB laptop (see the `hs_err_pid*.log` crash reports).

In [25]:
 # --- clear any dead Spark/Java session left over from a crash ---
from pyspark import SparkContext
from pyspark.sql import SparkSession
try:
    spark.stop()
except Exception:
    pass
SparkContext._active_spark_context = None
SparkContext._gateway = None
SparkContext._jvm = None
SparkSession._instantiatedSession = None
SparkSession._activeSession = None
# ----------------------------------------------------------------

from pyspark.sql import types as T
from pyspark.sql.window import Window

DRIVER_MEMORY      = os.environ.get("CW_DRIVER_MEMORY", "4g")
SHUFFLE_PARTITIONS = int(os.environ.get("CW_SHUFFLE_PARTITIONS", "32"))
LOCAL_DIR          = os.environ.get("CW_LOCAL_DIR", "E:/spark-tmp")

spark = (SparkSession.builder
         .appName("7006SCN-Task1-GHCN-Regression")
         .master("local[*]")
         .config("spark.driver.memory", DRIVER_MEMORY)
         .config("spark.local.dir", LOCAL_DIR)
         .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
         .config("spark.sql.ansi.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

CLUSTER_CONFIG = {
    "spark_version": spark.version,
    "master": spark.sparkContext.master,
    "default_parallelism": spark.sparkContext.defaultParallelism,
    "driver_memory": DRIVER_MEMORY,
    "shuffle_partitions": SHUFFLE_PARTITIONS,
    "local_dir": LOCAL_DIR,
    "ansi_enabled": spark.conf.get("spark.sql.ansi.enabled"),
}
for k, v in CLUSTER_CONFIG.items():
    print(f"{k:20s}: {v}")
TIMINGS = {}

spark_version       : 4.2.0
master              : local[*]
default_parallelism : 12
driver_memory       : 4g
shuffle_partitions  : 32
local_dir           : E:/spark-tmp
ansi_enabled        : true


3. dataset  loading

In [26]:
DAY_COLS = [f"DAY{i}" for i in range(1, 32)]

RAW_SCHEMA = T.StructType(                                       # EXPLAIN: StructType = list of columns
    [T.StructField("ID", T.StringType(), True),                  # EXPLAIN: StructField(name, type, nullable)
     T.StructField("YEAR", T.IntegerType(), True),
     T.StructField("MONTH", T.IntegerType(), True),
     T.StructField("ELEM", T.StringType(), True)]
    + [T.StructField(d, T.DoubleType(), True) for d in DAY_COLS])

raw = (spark.read
       .option("header", "true")                                 # EXPLAIN: first line holds the column names
       .option("mode", "PERMISSIVE")                             # EXPLAIN: bad values become NULL instead of failing
       .schema(RAW_SCHEMA)                                       # EXPLAIN: use the declared types
       .csv(CSV_PATH))                                           # EXPLAIN: lazy; nothing is read until an action runs
raw.show(6, truncate=True)                                       # EXPLAIN: print the first 6 rows

+-----------+----+-----+----+----+-----+-----+-----+-----+-----+------+------+------+------+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|         ID|YEAR|MONTH|ELEM|DAY1| DAY2| DAY3| DAY4| DAY5| DAY6|  DAY7|  DAY8|  DAY9| DAY10| DAY11|DAY12|DAY13|DAY14|DAY15|DAY16|DAY17|DAY18|DAY19|DAY20|DAY21|DAY22|DAY23|DAY24|DAY25|DAY26|DAY27|DAY28|DAY29|DAY30|DAY31|
+-----------+----+-----+----+----+-----+-----+-----+-----+-----+------+------+------+------+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|USC00010160|1970|    1|TMAX|39.0| 72.0|122.0| 61.0|106.0|100.0|  44.0| -39.0| -22.0| -11.0|  39.0| 44.0| 61.0|122.0|150.0|128.0|156.0|100.0|117.0| 28.0| 17.0| 11.0| 39.0| 33.0|150.0|178.0|228.0|228.0|222.0|189.0| 89.0|
|USC00010160|1970|    1|TMIN|11.0|-61.0|-39.0|-67.0|-50.0| -6.0|-111.0|-122.0|-150.0|-122.0|-111.0|  0.0|-33.0|-67.0|-39

4.SCHEMA INSPECTION

In [ ]:
# REF: Week1_Day02_PySpark_Student_Workbook › Exercise 2, Sub-task 2.1 (cell-9): df.columns
# ==============================================================================
raw.printSchema()                                                # EXPLAIN: prints every column name and type
ELEM_CODES = [r["ELEM"] for r in raw.select("ELEM").distinct().orderBy("ELEM").collect()]
print(f"Element codes (ELEM): {len(ELEM_CODES)} distinct ->", ELEM_CODES)   # VARIETY evidence

5. ROW AND COLUMN COUNT

In [28]:
# REF: Day02_LiveCoding › Block 0 activity (cell-6): rows = df.count(); len(df.columns
t0 = time.time()
N_ROWS = raw.count()                                             # EXPLAIN: ACTION - reads the whole file and counts rows
TIMINGS["full_scan_count_s"] = round(time.time() - t0, 2)        # EXPLAIN: time.time() = current time in seconds
N_COLS = len(raw.columns)                                        # EXPLAIN: raw.columns is the list of column names

print(f"rows    = {N_ROWS:,}")
print(f"columns = {N_COLS}")
print(f"full scan took {TIMINGS['full_scan_count_s']} s  ({N_ROWS / max(TIMINGS['full_scan_count_s'], 1e-6):,.0f} rows/s)")
print("Big Data requirement >= 10 million rows:", N_ROWS >= 10_000_000, "| >= 10 columns:", N_COLS >= 10)

rows    = 17,097,453
columns = 35
full scan took 5.34 s  (3,201,770 rows/s)
Big Data requirement >= 10 million rows: True | >= 10 columns: True


6. FILE SIZE VERIFICATION

In [29]:
# REFERENCE FROME  WEEK1 DAY2 FILE,EXECISE4 SUBTASK 4.3
def get_folder_size_mb(path):
    # EXPLAIN: returns the size in MB of one file, or of every file inside a folder
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 ** 2)               # EXPLAIN: getsize = bytes on disk
    total = 0
    for root, _, files in os.walk(path):                         # EXPLAIN: walk visits every sub-folder
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 ** 2)

FILE_BYTES = os.path.getsize(CSV_PATH)
FILE_MB    = get_folder_size_mb(CSV_PATH)
FILE_GIB   = FILE_BYTES / 1024 ** 3
print(f"climate.csv = {FILE_BYTES:,} bytes = {FILE_MB:,.1f} MB = {FILE_GIB:.2f} GiB")
print("Big Data requirement >= 1 GiB:", FILE_GIB >= 1)

climate.csv = 2,730,667,293 bytes = 2,604.2 MB = 2.54 GiB
Big Data requirement >= 1 GiB: True


7. PARTITION COUNT

In [30]:
 # REF: Day02_LiveCoding Block 0 activity (cell-6): df.rdd.getNumPartitions()
N_PARTITIONS = raw.rdd.getNumPartitions()          # EXPLAIN: how many pieces Spark split the file into
MAX_SPLIT_MB = int(spark.conf.get("spark.sql.files.maxPartitionBytes").rstrip("b")) / 1024 ** 2 \
    if spark.conf.get("spark.sql.files.maxPartitionBytes").rstrip("b").isdigit() else 128
print(f"partitions after ingestion = {N_PARTITIONS}")
print(f"average partition size     = {FILE_MB / N_PARTITIONS:,.1f} MB (Spark's split limit is 128 MB)")
print(f"parallel task slots        = {spark.sparkContext.defaultParallelism} -> "
      f"{math.ceil(N_PARTITIONS / spark.sparkContext.defaultParallelism)} waves of tasks per full scan")

partitions after ingestion = 21
average partition size     = 124.0 MB (Spark's split limit is 128 MB)
parallel task slots        = 12 -> 2 waves of tasks per full scan


8. Data-quality profile: evidence for the 5 V's.

In [32]:
from pyspark.sql import functions as F
# REF: Week1_Day02_PySpark_Student_Workbook › Exercise 3, Sub-tasks 3.1–3.3
VALID_RANGE = {"TMAX": (-600, 600), "TMIN": (-600, 600), "TOBS": (-600, 600),
               "PRCP": (0, 11000), "SNOW": (0, 2000), "SNWD": (0, 12000)}

# 8.1 Completeness: % NULL in every DAY column, all computed in ONE pass over the data
null_pct = raw.select([(F.count(F.when(F.col(c).isNull(), c)) / N_ROWS * 100).alias(c)
                       for c in DAY_COLS]).collect()[0].asDict()
# EXPLAIN: F.when(cond, c) keeps the value only when cond is true; F.count counts the non-NULL results
print(f"COMPLETENESS  NULL % DAY1-28 = {min(null_pct[f'DAY{i}'] for i in range(1, 29)):.2f}-"
      f"{max(null_pct[f'DAY{i}'] for i in range(1, 29)):.2f}%, DAY29 = {null_pct['DAY29']:.2f}%, "
      f"DAY30 = {null_pct['DAY30']:.2f}%, DAY31 = {null_pct['DAY31']:.2f}% (short months have no day 29-31)")

# 8.2 Uniqueness: one row is expected per (ID, YEAR, MONTH, ELEM)
dup_rows = N_ROWS - raw.select("ID", "YEAR", "MONTH", "ELEM").distinct().count()
print(f"UNIQUENESS    duplicate (ID,YEAR,MONTH,ELEM) rows = {dup_rows:,}")

# 8.3 Validity per element (Table T1.1)
lo = F.coalesce(*[F.when(F.col("ELEM") == e, F.lit(r[0])) for e, r in VALID_RANGE.items()])  # EXPLAIN: first non-NULL
hi = F.coalesce(*[F.when(F.col("ELEM") == e, F.lit(r[1])) for e, r in VALID_RANGE.items()])
prof = (raw.withColumn("lo", lo).withColumn("hi", hi)            # EXPLAIN: withColumn adds a column
        .groupBy("ELEM")                                         # EXPLAIN: one result row per element
        .agg(F.count("*").alias("rows"),
             sum(F.sum(F.col(c).isNotNull().cast("int")) for c in DAY_COLS).alias("values_present"),
             sum(F.sum((F.col(c).isNotNull() & ((F.col(c) < F.col("lo")) | (F.col(c) > F.col("hi")))).cast("int"))
                 for c in DAY_COLS).alias("values_invalid"),
             sum(F.sum((F.col(c) == 32767).cast("int")) for c in DAY_COLS).alias("values_32767"))
        .withColumn("invalid_pct", F.round(F.col("values_invalid") / F.col("values_present") * 100, 4))
        .orderBy("ELEM"))
print("VALIDITY      Table T1.1")

COMPLETENESS  NULL % DAY1-28 = 14.16-14.60%, DAY29 = 19.78%, DAY30 = 21.53%, DAY31 = 50.29% (short months have no day 29-31)
UNIQUENESS    duplicate (ID,YEAR,MONTH,ELEM) rows = 0
VALIDITY      Table T1.1


In [ ]:
# 8.3 (continued) Table T1.1 - validity of the six elements used in this project
## AI: display and summary written with AI assistance
t0 = time.time()
T1_1 = prof.filter(F.col("ELEM").isin(list(VALID_RANGE))).toPandas()   # EXPLAIN: 6 rows, small enough for pandas
TIMINGS["validity_profile_s"] = round(time.time() - t0, 2)
print("Table T1.1  Validity per element (value outside its physical range, or the 32767 error code)")
print(T1_1.to_string(index=False))

VALIDITY = {r.ELEM: {"values_present": int(r.values_present), "values_invalid": int(r.values_invalid),
                     "values_32767": int(r.values_32767), "invalid_pct": float(r.invalid_pct)}
            for r in T1_1.itertuples()}
INVALID_PCT_ALL = round(100 * float(T1_1.values_invalid.sum()) / float(T1_1.values_present.sum()), 4)
print(f"\nOverall invalid share across the six elements: {INVALID_PCT_ALL}%")

In [ ]:
 # AI: plotting code written with AI assistance
import matplotlib.pyplot as plt
FIG_DIR = "../figures" if os.path.basename(os.getcwd()).lower() == "notebooks" else "figures"
os.makedirs(FIG_DIR, exist_ok=True)                             # EXPLAIN: repo-root figures/ folder
fig, ax = plt.subplots(figsize=(10, 3))                          # EXPLAIN: create an empty figure
ax.bar(range(1, 32), [null_pct[c] for c in DAY_COLS])            # EXPLAIN: one bar per DAY column
ax.set_xlabel("DAY column"); ax.set_ylabel("% NULL"); ax.set_xticks(range(1, 32))
ax.set_title("Fig. T1.1  Missing values per DAY column")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/T1_1_completeness.png", dpi=150); plt.show()

 9. Preprocessing and feature engineering

REF: Week1_Day02_PySpark_Student_Workbook › Exercise 2.3 · AI: element-join + single `stack()` reshape, gap-aware lags and the target written with AI assistance

### Data preparation decisions (in the order the code runs)

| Step | Decision | Why (and the alternative I rejected) |
|---|---|---|
| Bronze layer | Keep the 6 elements used and save them once as Parquet | The CSV is parsed once; later steps read compressed columns. Re-reading the 2.54 GiB CSV every time was rejected as too slow. |
| Station sample | Keep a repeatable 5 % of stations (`crc32(ID) % 100 < 5`) | Spark ran out of memory on my 16 GB laptop with 100 %. Whole stations are kept, so each time series stays complete; a random row sample was rejected because it would break the lags. |
| Duplicates | `dropDuplicates` on (ID, YEAR, MONTH, ELEM) | One value per station-month-element; section 8 found 0 duplicates, so this is a safety check. |
| Reshape | `stack` 31 DAY columns into rows, then `pivot` ELEM back into columns | One shuffle instead of 6 filters + 5 joins; gives one row per station-day. |
| Validity | Drop values outside each element's physical range and the 32767 code; drop dates like 31 Feb; set TMIN > TMAX to NULL | Table T1.1 shows these values exist; keeping them would corrupt both features and target. |
| Units | Divide TMAX, TMIN, TOBS, PRCP by 10 | GHCN stores tenths; the target is reported in °C. |
| Lag features | `TMAX_lag1/2` only when the earlier row is exactly 1 or 2 calendar days back | A plain `lag()` would jump across gaps and use a value from weeks earlier. |
| Rolling / season | 7-day means, daily range, `sin/cos` of day-of-year, year, state code | Captures recent warmth, season and coarse location; sin/cos keeps 31 Dec next to 1 Jan. |
| Target | `label_TMAX_next` only when the next row is exactly 1 day later | No target is invented across a gap. |
| Split | Train 1970–2015, test 2016–2023 (by time) | A random split was rejected: neighbouring days are near-identical, so it would leak the future into training. |

In [36]:
  # REF: Week1_Day02_PySpark_Student_Workbook › Exercise 2.3 (cell-9)
## AI: bronze Parquet cache, single-pass unpivot + pivot (replaces 6 filters + 5 joins), gap-aware lags,
## target and Spark auto-restart written with AI assistance
# ==============================================================================
from pyspark import SparkContext
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

# ---- 9.0a Make sure Spark is alive; if the Java process died, start a fresh one ----
def spark_alive():
    try:
        spark.range(1).count()
        return True
    except Exception:
        return False

if not spark_alive():
    print("Spark was not running - starting a fresh session ...")
    SparkContext._active_spark_context = None
    SparkContext._gateway = None
    SparkContext._jvm = None
    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    spark = (SparkSession.builder
             .appName("7006SCN-Task1-GHCN-Regression")
             .master("local[*]")
             .config("spark.driver.memory", os.environ.get("CW_DRIVER_MEMORY", "4g"))
             .config("spark.local.dir", os.environ.get("CW_LOCAL_DIR", "E:/spark-tmp"))
             .config("spark.sql.shuffle.partitions", "16")
             .config("spark.sql.ansi.enabled", "true")
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    raw = (spark.read.option("header", "true").option("mode", "PERMISSIVE")
           .schema(RAW_SCHEMA).csv(CSV_PATH))                                    # EXPLAIN: re-link raw to the new session

# ---- 9.0b Settings ----
ELEMS = ["TMAX", "TMIN", "TOBS", "PRCP", "SNOW", "SNWD"]
KEYS  = ["ID", "YEAR", "MONTH"]
LAKE  = os.environ.get("CW_LAKE_DIR", "/content/lakehouse" if IN_COLAB else "./lakehouse")
VALID_RANGE = {"TMAX": (-600, 600), "TMIN": (-600, 600), "TOBS": (-600, 600),
               "PRCP": (0, 11000), "SNOW": (0, 2000), "SNWD": (0, 12000)}

spark.conf.set("spark.sql.shuffle.partitions", "16")                            # EXPLAIN: fewer, larger shuffle partitions for a laptop
spark.conf.set("spark.sql.adaptive.enabled", "true")                            # EXPLAIN: AQE merges small partitions

# Repeatable 5% whole-station sample (crc32 hash of ID): keeps each station's full time series,
# and fits the 16 GB laptop used for this run. Same sample as the pipeline in section 10.
STATION_SAMPLE_PCT = int(os.environ.get("CW_STATION_SAMPLE_PCT", "5"))

# ---- 9.0 BRONZE: parse the CSV once, keep only the 6 elements we use ----
BRONZE = f"{LAKE}/bronze_raw"
if not os.path.exists(f"{BRONZE}/_SUCCESS"):
    t0 = time.time()
    raw.filter(F.col("ELEM").isin(ELEMS)).write.mode("overwrite").parquet(BRONZE)
    TIMINGS["bronze_build_s"] = round(time.time() - t0, 2)
    print(f"bronze written in {TIMINGS['bronze_build_s']} s")
bronze = spark.read.parquet(BRONZE)

base = bronze
if STATION_SAMPLE_PCT < 100:
    base = base.filter(F.crc32("ID") % 100 < STATION_SAMPLE_PCT)                # EXPLAIN: crc32 = repeatable hash
base = base.dropDuplicates(["ID", "YEAR", "MONTH", "ELEM"])                       # 9.1 one row per key

# ---- 9.2-9.4 unpivot DAY1..DAY31 -> rows, drop invalid values and impossible dates ----
stack_sql = "stack(31, " + ", ".join(f"{i}, `{d}`" for i, d in enumerate(DAY_COLS, start=1)) + ") as (DAY, VAL)"

valid = None
for e in ELEMS:
    lo_, hi_ = VALID_RANGE[e]
    c = (F.col("ELEM") == e) & F.col("VAL").between(lo_, hi_)
    valid = c if valid is None else (valid | c)
valid = valid & (F.col("VAL") != 32767)

days_in_month = F.dayofmonth(F.last_day(F.make_date("YEAR", "MONTH", F.lit(1))))
long = (base.select(*KEYS, "ELEM", F.expr(stack_sql))
            .filter(valid)
            .filter(F.col("DAY") <= days_in_month))                              # 9.4 drop e.g. 31 Feb

# ---- 9.5 pivot back to one row per station-day, convert units ----
daily = (long.groupBy(*KEYS, "DAY")
             .pivot("ELEM", ELEMS)                                               # EXPLAIN: explicit list = no extra job
             .agg(F.first("VAL"))
             .filter(F.col("TMAX").isNotNull())                                  # TMAX is needed for the target
             .withColumn("date", F.make_date("YEAR", "MONTH", "DAY"))
             .withColumn("TMAX", F.col("TMAX") / 10).withColumn("TMIN", F.col("TMIN") / 10)
             .withColumn("TOBS", F.col("TOBS") / 10).withColumn("PRCP", F.col("PRCP") / 10)
             .withColumn("TMIN", F.when(F.col("TMIN") <= F.col("TMAX"), F.col("TMIN"))))  # TMIN above TMAX is an error

# ---- 9.6 / 9.7 features and target, per station in date order ----
w      = Window.partitionBy("ID").orderBy("dnum")
w_7day = Window.partitionBy("ID").orderBy("dnum").rangeBetween(-6, 0)            # EXPLAIN: today + 6 previous days

def lag_exact(col, k):
    # EXPLAIN: value from k days earlier, only if that row is exactly k calendar days back (no leakage across gaps)
    return F.when(F.lag("dnum", k).over(w) == F.col("dnum") - k, F.lag(col, k).over(w))

features = (daily
    .withColumn("dnum", F.datediff("date", F.lit("1970-01-01").cast("date")))
    .withColumn("TMAX_lag1", lag_exact("TMAX", 1))
    .withColumn("TMAX_lag2", lag_exact("TMAX", 2))
    .withColumn("TMAX_roll7", F.avg("TMAX").over(w_7day))
    .withColumn("TMIN_roll7", F.avg("TMIN").over(w_7day))
    .withColumn("DTR", F.col("TMAX") - F.col("TMIN"))
    .withColumn("doy_sin", F.sin(2 * math.pi * F.dayofyear("date") / 365.25))
    .withColumn("doy_cos", F.cos(2 * math.pi * F.dayofyear("date") / 365.25))
    .withColumn("YEAR_f", F.col("YEAR").cast("double"))
    .withColumn("state_code", F.when(F.substring("ID", 1, 3) == "USC", F.substring("ID", 6, 2)).otherwise("OTHER"))
    .withColumn("label_TMAX_next",                                                # TARGET
                F.when(F.lead("dnum", 1).over(w) == F.col("dnum") + 1, F.lead("TMAX", 1).over(w)))
    .filter(F.col("label_TMAX_next").isNotNull())
    .drop("dnum", "DAY"))

# ---- write SILVER, partitioned by YEAR ----
t0 = time.time()
(features
    .sortWithinPartitions("YEAR")
    .write.mode("overwrite").partitionBy("YEAR").parquet(f"{LAKE}/silver_features"))
TIMINGS["feature_build_s"] = round(time.time() - t0, 2)

silver = spark.read.parquet(f"{LAKE}/silver_features")
N_SILVER = silver.count()
print(f"station-day rows after preprocessing ({STATION_SAMPLE_PCT}% station sample): {N_SILVER:,} | "
      f"built in {TIMINGS['feature_build_s']} s | Parquet size {get_folder_size_mb(f'{LAKE}/silver_features'):,.1f} MB")
silver.select("ID", "date", "TMAX", "TMAX_lag1", "TMAX_roll7", "state_code", "label_TMAX_next").show(5)

NUM_FEATURES = ["TMAX", "TMIN", "TOBS", "PRCP", "SNOW", "SNWD", "TMAX_lag1", "TMAX_lag2",
                "TMAX_roll7", "TMIN_roll7", "DTR", "doy_sin", "doy_cos", "YEAR_f"]
CORR = {c: round(v, 4) for c, v in silver.select([F.corr(c, "label_TMAX_next").alias(c) for c in NUM_FEATURES])
        .collect()[0].asDict().items() if v is not None}
print("Table T1.2  correlation with next-day TMAX:", dict(sorted(CORR.items(), key=lambda kv: -abs(kv[1]))))

Spark was not running - starting a fresh session ...
station-day rows after preprocessing (5% station sample): 2,989,300 | built in 159.37 s | Parquet size 68.9 MB
+-----------+----------+----+---------+------------------+----------+---------------+
|         ID|      date|TMAX|TMAX_lag1|        TMAX_roll7|state_code|label_TMAX_next|
+-----------+----------+----+---------+------------------+----------+---------------+
|USC00029359|1984-01-01|12.2|     12.2| 9.214285714285714|        02|           10.6|
|USC00029359|1984-01-02|10.6|     12.2| 9.928571428571429|        02|           12.2|
|USC00029359|1984-01-03|12.2|     10.6|10.714285714285714|        02|           13.9|
|USC00029359|1984-01-04|13.9|     12.2|12.142857142857142|        02|           12.8|
|USC00029359|1984-01-05|12.8|     13.9|12.542857142857143|        02|           11.7|
+-----------+----------+----+---------+------------------+----------+---------------+
only showing top 5 rows
Table T1.2  correlation with next-day 

### 9.8 Five V's evidence — Table T1.3
Every value below is computed in this notebook, not typed in.

In [ ]:
# 9.8 Table T1.3 - the five V's, each backed by a figure measured in THIS notebook
## AI: table layout written with AI assistance; every value is a variable computed above
n_elem   = len(globals().get("ELEM_CODES", [])) or "see section 4"
inv_pct  = globals().get("INVALID_PCT_ALL", "run Table T1.1")
full_est = N_SILVER * 100 / STATION_SAMPLE_PCT
FIVE_VS = {
    "Volume":   f"{N_ROWS:,} rows x {N_COLS} columns; {FILE_GIB:.2f} GiB; {N_PARTITIONS} input partitions of "
                f"~{FILE_MB / N_PARTITIONS:,.0f} MB; ~{full_est / 1e6:,.0f} M station-day rows at full scale "
                f"({N_SILVER:,} in the {STATION_SAMPLE_PCT}% sample)",
    "Velocity": f"historical batch (NOAA updates the source daily); full scan {TIMINGS['full_scan_count_s']} s = "
                f"{N_ROWS / TIMINGS['full_scan_count_s']:,.0f} rows/s on {spark.sparkContext.defaultParallelism} cores; "
                f"CSV parsed once into bronze Parquet",
    "Variety":  f"{n_elem} element codes in one ELEM column; 31 wide DAY columns; values in tenths of deg C / mm; "
                f"reshaped to one row per station-day",
    "Veracity": f"NULL {min(null_pct[f'DAY{i}'] for i in range(1, 29)):.2f}-{max(null_pct[f'DAY{i}'] for i in range(1, 29)):.2f}% "
                f"for DAY1-28 vs {null_pct['DAY31']:.2f}% for DAY31; {dup_rows:,} duplicate keys; "
                f"{inv_pct}% invalid values in the 6 elements used (Table T1.1)",
    "Value":    f"next-day TMAX correlates r = {CORR['TMAX']} with today's TMAX and r = {CORR['TMAX_lag1']} "
                f"with yesterday's (Table T1.2) - a strong, usable signal for regression",
}
print("Table T1.3  Five V's of this dataset")
for v, text in FIVE_VS.items():
    print(f"  {v:9s} | {text}")
print("\nDominant V's: Volume (the reshape multiplies rows and exceeded my laptop's memory) and "
      "Veracity (gaps, sentinels and short months directly corrupt lags and target).")

### 9.9 Licence and ethical note
- **Licence:** the Hugging Face mirror declares no licence. The underlying NOAA GHCN-Daily data is US-Government public-domain data, and NOAA asks users to cite Menne et al. (2012). Because the mirror's provenance is not verified, results are attributed to NOAA and the mirror is treated as a convenience copy.
- **Privacy / consent:** no personal data — every record describes a weather station, not a person.
- **Bias risk:** station density is uneven across states and decades, so prediction error may be higher in sparsely covered (often rural) areas. Error per state is checked in Task 3.

## 10. PySpark Pipeline construction

| Order | Stage | Transformer | Why this stage | Why it must be here |
|---|---|---|---|---|
| 1 | Impute | `Imputer(strategy="median")` | Lags are NULL after gaps; SNOW/SNWD are often missing. Median, not mean, because PRCP and SNOW are heavily right-skewed. | `VectorAssembler` rejects NULLs, so imputation must come before it. |
| 2 | Index state | `StringIndexer(handleInvalid="keep")` | Turns the state code into a number; `keep` stops an unseen state crashing the test set. | `OneHotEncoder` only accepts numeric indices, so indexing comes first. |
| 3 | Encode state | `OneHotEncoder` | Sparse 0/1 vector, so models do not read a false order into state numbers. | Needs the index from stage 2. |
| 4 | Assemble | `VectorAssembler` | Spark ML models need one `features` vector column. | Needs the imputed numbers (1) and the state vector (3). |
| 5 | Scale | `StandardScaler(withMean=False)` | Unit variance, so SNWD (mm) does not dominate `doy_sin` (−1…1); `withMean=False` keeps the vector sparse. | Works on a vector, so it must follow the assembler. |

**What breaks if two are swapped:** Assembler before Imputer → the fit fails on the first NULL lag. Encoder before Indexer → type error (string input). Scaler before Assembler → no vector to scale.
**Leakage control:** all five stages are fitted on the 1970–2015 training rows only, so medians, state indices and scales contain nothing from the 2016–2023 test years.

In [40]:
# 10.1 Train/test split BY TIME (not random: neighbouring days are near-identical and would leak)
# The silver features were already built on the repeatable station sample (section 9);
# this filter uses the same crc32 rule, so it keeps exactly the same stations.
silver_subset = silver.filter(F.crc32("ID") % 100 < STATION_SAMPLE_PCT)

TRAIN_MAX_YEAR = 2015                                                    # EXPLAIN: train 1970-2015, test 2016-2023 (time-based split, no leakage)
train_df_sampled = silver_subset.filter(F.col("YEAR") <= TRAIN_MAX_YEAR)
test_df_sampled  = silver_subset.filter(F.col("YEAR") >  TRAIN_MAX_YEAR)
print(f"Sampled Train split: {train_df_sampled.count():,} rows, Sampled Test split: {test_df_sampled.count():,} rows")

from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

NUMERIC_COLS  = NUM_FEATURES
IMPUTED_COLS  = [f"{c}_imputed" for c in NUMERIC_COLS]

# Stage 1: Imputer
imputer = Imputer(
    strategy="median",
    inputCols=NUMERIC_COLS,
    outputCols=IMPUTED_COLS
)

# Stage 2: StringIndexer
indexer = StringIndexer(
    handleInvalid="keep",
    inputCols=["state_code"],
    outputCols=["state_code_indexed"]
)

# Stage 3: OneHotEncoder
encoder = OneHotEncoder(
    inputCols=["state_code_indexed"],
    outputCols=["state_code_vec"]
)

# Stage 4: VectorAssembler
assembler = VectorAssembler(
    inputCols=IMPUTED_COLS + ["state_code_vec"],
    outputCol="raw_features"
)

# Stage 5: StandardScaler
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=False,
    withStd=True
)

# Complete Pipeline
pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler])

print("Fitting PySpark Pipeline Model...")
t0 = time.time()
fitted_model = pipeline.fit(train_df_sampled)
TIMINGS["pipeline_fit_s"] = round(time.time() - t0, 2)

# Deliverable Stage List Inspection
stages_list = [type(s).__name__ for s in fitted_model.stages]
print(f"Fitted Pipeline Stages: {stages_list}")
PIPELINE_STAGES = stages_list

# Check the output on 3 rows
prepared = fitted_model.transform(train_df_sampled.limit(3))
prepared.select("ID", "date", "features", "label_TMAX_next").show(3, truncate=70)
print("feature vector length:", prepared.first()["features"].size, "| fit time:", TIMINGS["pipeline_fit_s"], "s")

# Save Fitted Pipeline Model
MODEL_DIR = os.environ.get("CW_MODEL_DIR", "/content/models" if IN_COLAB else "./models")
model_save_path = f"{MODEL_DIR}/task1_pipeline"
fitted_model.write().overwrite().save(model_save_path)
print(f"Fitted Pipeline Model saved to: {model_save_path}")

Sampled Train split: 2,555,512 rows, Sampled Test split: 433,788 rows
Fitting PySpark Pipeline Model...
Fitted Pipeline Stages: ['ImputerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'StandardScalerModel']
+-----------+----------+----------------------------------------------------------------------+---------------+
|         ID|      date|                                                              features|label_TMAX_next|
+-----------+----------+----------------------------------------------------------------------+---------------+
|USC00029359|1984-01-01|(54,[0,2,6,7,8,9,10,11,12,13,20],[1.021138947603382,0.4137180563126...|           10.6|
|USC00029359|1984-01-02|(54,[0,1,2,6,7,8,9,10,11,12,13,20],[0.8872190856226108,-0.560862895...|           12.2|
|USC00029359|1984-01-03|(54,[0,1,2,6,7,8,9,10,11,12,13,20],[1.021138947603382,-0.5148905271...|           13.9|
+-----------+----------+----------------------------------------------------------------------+--

## 11. Evidence pack EP1 (one cell, in the order the answer sheet requires)

In [ ]:
 # ==============================================================================
# EP1 - one cell, in the order required by the answer sheet
## AI: layout written with AI assistance
# Run this AFTER the pipeline has been fitted (fitted_model must exist)
# ==============================================================================
print("=" * 80)
print(f"7006SCN | {DECLARATION['student_name']} | {DECLARATION['student_email']} | "
      f"Pool ref {DECLARATION['pool_reference']} | {DECLARATION['dataset_name']}")
print("=" * 80)

spark = SparkSession.builder.appName("7006SCN-Task1-GHCN-Regression").getOrCreate()
print(f"[1] SparkSession : Spark {spark.version} | master {spark.sparkContext.master} | "
      f"{spark.sparkContext.defaultParallelism} cores | driver {DRIVER_MEMORY} | "
      f"shuffle partitions {spark.conf.get('spark.sql.shuffle.partitions')}")

print("[2] df.printSchema():")
raw.printSchema()

print(f"[3] rows = {raw.count():,} | columns = {len(raw.columns)}")
print(f"[4] file size = {os.path.getsize(CSV_PATH):,} bytes = "
      f"{os.path.getsize(CSV_PATH) / 1024**3:.2f} GiB")
print(f"[5] partitions after ingestion = {raw.rdd.getNumPartitions()}")

if "fitted_model" in globals():
    print(f"[6] Fitted Pipeline Stages: {[type(s).__name__ for s in fitted_model.stages]}")
else:
    print("[6] Fitted Pipeline Stages: NOT AVAILABLE - run the model training cell first, then rerun this cell")

## 12. `results/task1.json` and the Task 1 → Task 2 handoff
Saved in the **repository root** as `/results/task1.json` (the file the brief and Task 2 read) and `/results/task1_results.json` (same content). The `handoff` block freezes the train/test split and the fitted preprocessing, so Task 2 applies them and never re-splits or re-fits.

In [ ]:
# ==============================================================================
# 12. results/task1.json - Pool reference, dataset size, source, licence, pipeline (>=3 stages),
#     cluster config, five V's, quality, and the handoff Task 2 reads
## AI: JSON structure and handoff contract written with AI assistance; all values come from this run
# ==============================================================================
import shutil

PROJ = os.environ.get("CW_PROJECT_DIR",
       os.path.abspath("..") if os.path.basename(os.getcwd()).lower() == "notebooks" else os.getcwd())
PROJ = PROJ.replace("\\", "/")                                   # EXPLAIN: repo root, same rule as Task 2
os.makedirs(f"{PROJ}/results", exist_ok=True)

# --- freeze the split and the fitted preprocessing (paths relative to the repo root) ---
TRAIN_REL, TEST_REL, PREP_REL = "lakehouse/handoff/train", "lakehouse/handoff/test", "models/task1_preprocessing"
KEEP = ["ID", "date", "YEAR", "state_code", "label_TMAX_next"] + NUM_FEATURES
train_df_sampled.select(*KEEP).write.mode("overwrite").parquet(f"{PROJ}/{TRAIN_REL}")
test_df_sampled.select(*KEEP).write.mode("overwrite").parquet(f"{PROJ}/{TEST_REL}")
fitted_model.write().overwrite().save(f"{PROJ}/{PREP_REL}")       # EXPLAIN: fitted on TRAIN years only
N_TRAIN = spark.read.parquet(f"{PROJ}/{TRAIN_REL}").count()
N_TEST  = spark.read.parquet(f"{PROJ}/{TEST_REL}").count()

QUALITY = {
    "completeness_pct": {
        "min_day1_28": round(min(null_pct[f"DAY{i}"] for i in range(1, 29)), 2),
        "max_day1_28": round(max(null_pct[f"DAY{i}"] for i in range(1, 29)), 2),
        "day29": round(null_pct["DAY29"], 2),
        "day30": round(null_pct["DAY30"], 2),
        "day31": round(null_pct["DAY31"], 2),
    },
    "uniqueness": {"duplicate_rows": dup_rows},
    "validity": globals().get("VALIDITY", "Table T1.1 not run"),
    "invalid_pct_six_elements": globals().get("INVALID_PCT_ALL"),
}

task1 = {
    "pool_reference": DECLARATION["pool_reference"],
    "student": {k: DECLARATION[k] for k in ("student_name", "student_email", "sid")},
    "dataset": {"name": DECLARATION["dataset_name"], "source_url": DECLARATION["source_url"],
                "original_source": DECLARATION["original_source"], "licence": DECLARATION["licence"],
                "rows": N_ROWS, "columns": N_COLS, "file_size_bytes": FILE_BYTES,
                "file_size_gib": round(FILE_GIB, 3), "partitions_after_ingestion": N_PARTITIONS,
                "element_codes": len(globals().get("ELEM_CODES", []))},
    "big_data_requirements": {"rows_ge_10M": N_ROWS >= 10_000_000, "cols_ge_10": N_COLS >= 10,
                              "size_ge_1GiB": FILE_GIB >= 1, "not_kaggle": True, "pipeline_in_pyspark": True},
    "problem": {"type": "regression", "target": "label_TMAX_next",
                "target_meaning": "next-day maximum temperature at the same station (deg C)",
                "train_years": f"1970-{TRAIN_MAX_YEAR}", "test_years": f"{TRAIN_MAX_YEAR + 1}-2023"},
    "ethics": DECLARATION.get("ethics"),
    "five_vs": globals().get("FIVE_VS"),
    "sampling": {"station_sample_pct": STATION_SAMPLE_PCT,
                 "method": "crc32(ID) % 100 < pct - repeatable, whole stations kept"},
    "cluster_config": CLUSTER_CONFIG,
    "quality": QUALITY,
    "station_day_rows": N_SILVER,
    "features": NUM_FEATURES + ["state_code (one-hot)"],
    "feature_target_correlation": CORR,
    "pipeline_stages": PIPELINE_STAGES,
    "pipeline_stage_count": len(PIPELINE_STAGES),
    "timings_s": TIMINGS,
    "handoff": {
        "target_col": "label_TMAX_next",
        "features_col": "features",
        "train_path": TRAIN_REL,
        "test_path": TEST_REL,
        "preprocessing_model_path": PREP_REL,
        "train_rows": N_TRAIN,
        "test_rows": N_TEST,
        "train_years": f"1970-{TRAIN_MAX_YEAR}",
        "test_years": f"{TRAIN_MAX_YEAR + 1}-2023",
        "pipeline_stages": PIPELINE_STAGES,
        "log_target": False,                     # EXPLAIN: temperature can be negative, so log1p is not valid
    },
}

T1_PATH = f"{PROJ}/results/task1.json"
if os.path.exists(T1_PATH) and "handoff" not in json.load(open(T1_PATH)):
    shutil.copy(T1_PATH, f"{PROJ}/results/task1_previous_run.json")   # EXPLAIN: keep the older file
for path in (T1_PATH, f"{PROJ}/results/task1_results.json"):        # task1.json = brief + Task 2; task1_results.json = repo layout
    with open(path, "w") as f:
        json.dump(task1, f, indent=2)                                 # EXPLAIN: write the dictionary as JSON
    print("saved:", path)
print(f"handoff: train {N_TRAIN:,} rows | test {N_TEST:,} rows | stages {PIPELINE_STAGES}")
print(json.dumps(task1, indent=2))